# 例 7.2 · GLD 与 GDX 的协整检验

用两只 ETF 的复权收盘价检验它们是否存在长期价格关系，再估计每股 GLD 对应的 GDX 对冲股数，画出这一组合的历史价差。原例见 [MATLAB](../matlab/example7_2.m) 和 [Python](../python/examples/example7_2.py)。

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

repo_root = next(
    base for base in (Path.cwd(), *Path.cwd().parents)
    if (base / 'python' / 'epchan').is_dir()
)
sys.path.insert(0, str(repo_root / 'python'))

from epchan.cadf import cadf
from epchan.get_tday import get_tday
from epchan.ols import ols
from epchan.paths import data_path
from epchan.prt import prt

## 1. 按共同交易日对齐价格

原例先取日期并集，再删除任一价格缺失的行。这里直接按日期交集取出配对价格，再剔除非有限值，得到相同的回归样本。

In [ ]:
gld_dates, gld_prices = get_tday(data_path('GLD.xls'))
gdx_dates, gdx_prices = get_tday(data_path('GDX.xls'))
dates, i, j = np.intersect1d(gld_dates, gdx_dates, return_indices=True)
gld, gdx = gld_prices[i], gdx_prices[j]
valid = np.isfinite(gld) & np.isfinite(gdx)
dates, gld, gdx = dates[valid], gld[valid], gdx[valid]
print(f'共同有效交易日共 {len(dates)} 天，日期从 {dates[0]} 到 {dates[-1]}。')

## 2. 对协整回归的残差做 CADF 检验

以 GLD 为因变量，GDX 为解释变量。`p=0` 表示协整回归含常数项，`l=1` 表示检验中加入一阶滞后残差差分。统计量越负，拒绝无协整原假设的证据越强。

In [ ]:
result = cadf(gld, gdx, p=0, l=1)
prt(result, ['GLD', 'GDX'])

## 3. 估计对冲比率并绘制价差

为复现书中图 7.4，这一步依照原例单独拟合**无截距**的回归 $GLD_t = \beta GDX_t + z_t$。做多 1 股 GLD、做空 $\beta$ 股 GDX，其价差就是残差 $z_t$。图中的零线表示两边按复权价格计量的价值相等。

In [ ]:
fit = ols(gld, gdx)
hedge_ratio = float(fit.beta[0, 0])
spread = fit.resid.ravel()
print(f'对冲比率 = {hedge_ratio:.4f} 股 GDX / 股 GLD')

In [ ]:
trading_days = [datetime.strptime(str(day), '%Y%m%d').date() for day in dates]
fig, ax = plt.subplots(figsize=(9, 3.6), layout='constrained')
ax.axhline(0, color='0.65', linewidth=0.8, linestyle='--')
ax.plot(trading_days, spread, color='#286BB0', linewidth=1.5)
ax.set(
    title=f'GLD minus {hedge_ratio:.4f} shares of GDX',
    xlabel='Trading date', ylabel='Spread (USD per GLD share)',
)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

### 结果解读

当前仓库数据的 CADF 统计量约为 **−3.518**，低于 5% 临界值 **−3.327**，高于 1% 临界值 **−3.902**。因此样本在 5% 水平支持 GLD 与 GDX 协整，证据尚未达到 1% 水平。对冲比率约为 **1.6395**。书中报告的统计量为 −3.357，对冲比率为 1.6766，检验在这两个显著性水平上的结论相同。